## RFM 기반 데이터 가공
- 이 데이터는 거래 데이터가 아니라 행동 데이터라서 클래식 RFM(구매 기반)이 아니라 Engagement-based RFM으로 만들고, Engagement 컬럼을 추가하였다.

|컬럼명|사용 컬럼|의미|
|---|---|---|
|activity_score|notifications_clicked|사용자의 알림 클릭 횟수를 기반으로 한 최근 활동 수준 지표.|
|adjusted_frequency|weekly_songs_played * (1 - song_skip_rate)|노래 재생 횟수에 스킵 비율을 반영한 실제 콘텐츠 소비 빈도 지표.|
|Monetary|weekly_hours|주간 청취 시간을 기반으로 한 서비스 이용 가치 지표.|
|Engagement|num_playlists_created + num_platform_friends + num_shared_friends|플레이리스트 생성, 친구 추가, 공유 활동을 합산한 플랫폼 참여도 지표.|
|subscription_risk |num_subscription_pauses|구독 일시정지 횟수를 기반으로 한 사용자 이탈 위험 신호 지표.|
|support_pressure|customer_service_inquiries|고객센터 문의 횟수를 기반으로 한 서비스 불만 또는 문제 경험 지표.|

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# new rfm_df

In [4]:
model_df = pd.read_csv("01_data/processed/model_df.csv")
model_df

,age,location,subscription_type,payment_plan,num_subscription_pauses,payment_method,customer_service_inquiries,weekly_hours,average_session_length,song_skip_rate,weekly_songs_played,weekly_unique_songs,num_favorite_artists,num_platform_friends,num_playlists_created,num_shared_playlists,notifications_clicked,churned,State_AvgIncome,tenure_days
0,32,Montana,Free,Yearly,2,Paypal,Medium,22.391362,1.756575,0.176873,169,109,18,32,52,35,46,0,47631.156827,1606
1,64,New Jersey,Free,Monthly,3,Paypal,Low,29.294210,0.875019,0.981811,55,55,44,33,12,25,37,1,76533.296517,2897
2,51,Washington,Premium,Yearly,2,Credit Card,High,15.400312,0.411728,0.048411,244,117,20,129,50,28,38,0,64452.058299,348
3,63,California,Family,Yearly,4,Apple Pay,Medium,22.842084,1.393258,0.035691,442,252,47,120,55,17,24,0,67180.181705,2894
4,54,Washington,Family,Monthly,3,Paypal,High,23.151163,0.876304,0.039738,243,230,41,66,40,32,47,0,64452.058299,92
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124995,32,California,Student,Monthly,4,Debit Card,Low,29.161399,0.119612,0.893839,460,22,18,199,71,41,41,0,67180.181705,1895
124996,56,Maine,Premium,Yearly,2,Debit Card,Low,44.744198,1.751156,0.668759,315,16,48,185,67,23,30,0,49140.932961,2778
124997,45,Maine,Family,Monthly,0,Paypal,Medium,15.200073,1.301067,0.286604,11,11,48,40,78,40,28,0,49140.932961,604
124998,69,Maine,Free,Monthly,4,Paypal,High,35.270053,0.348684,0.092528,451,108,15,41,3,35,9,1,49140.932961,2570


In [5]:
rfm_df = model_df.assign(
    activity_score = model_df["notifications_clicked"],
    adjusted_frequency = model_df["weekly_songs_played"] * (1 - model_df["song_skip_rate"]),
    Monetary = model_df["weekly_hours"],
    Engagement = (
        model_df["num_playlists_created"] +
        model_df["num_platform_friends"] +
        model_df["num_shared_playlists"]
    ),
    subscription_risk = model_df["num_subscription_pauses"],
    support_pressure = model_df["customer_service_inquiries"]
)[[
    "activity_score",
    "adjusted_frequency",
    "Monetary",
    "Engagement",
    "subscription_risk",
    "support_pressure"
]]
rfm_df

,activity_score,adjusted_frequency,Monetary,Engagement,subscription_risk,support_pressure
0,46,139.108484,22.391362,119,2,Medium
1,37,1.000403,29.294210,70,3,Low
2,38,232.187706,15.400312,207,2,High
3,24,426.224473,22.842084,192,4,Medium
4,47,233.343691,23.151163,138,3,High
...,...,...,...,...,...,...
124995,41,48.833931,29.161399,311,4,Low
124996,30,104.340966,44.744198,275,2,Low
124997,28,7.847359,15.200073,158,0,Medium
124998,9,409.269716,35.270053,79,4,High


In [6]:
# 숫자형 인코딩
rfm_df.info()

mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2,
}

rfm_df['support_pressure'] = rfm_df['support_pressure'].map(mapping)

rfm_df

<class 'pandas.DataFrame'>
RangeIndex: 125000 entries, 0 to 124999
Data columns (total 6 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   activity_score      125000 non-null  int64  
 1   adjusted_frequency  125000 non-null  float64
 2   Monetary            125000 non-null  float64
 3   Engagement          125000 non-null  int64  
 4   subscription_risk   125000 non-null  int64  
 5   support_pressure    125000 non-null  str    
dtypes: float64(2), int64(3), str(1)
memory usage: 5.7 MB


,activity_score,adjusted_frequency,Monetary,Engagement,subscription_risk,support_pressure
0,46,139.108484,22.391362,119,2,1
1,37,1.000403,29.294210,70,3,0
2,38,232.187706,15.400312,207,2,2
3,24,426.224473,22.842084,192,4,1
4,47,233.343691,23.151163,138,3,2
...,...,...,...,...,...,...
124995,41,48.833931,29.161399,311,4,0
124996,30,104.340966,44.744198,275,2,0
124997,28,7.847359,15.200073,158,0,1
124998,9,409.269716,35.270053,79,4,2


In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

rfm_scaled_df = scaler.fit_transform(rfm_df)
rfm_scaled_df = pd.DataFrame(rfm_scaled_df, columns=rfm_df.columns, index=rfm_df.index)
rfm_scaled_df

,activity_score,adjusted_frequency,Monetary,Engagement,subscription_risk,support_pressure
0,1.494381,0.125060,-0.183121,-0.828193,0.006249,0.002839
1,0.870369,-1.129199,0.294670,-1.569739,0.711868,-1.221010
2,0.939704,0.970380,-0.667016,0.503562,0.006249,1.226689
3,-0.030982,2.732568,-0.151923,0.276558,1.417487,0.002839
4,1.563716,0.980878,-0.130530,-0.540655,0.711868,1.226689
...,...,...,...,...,...,...
124995,1.147708,-0.694788,0.285477,2.077455,1.417487,-1.221010
124996,0.385026,-0.190689,1.364064,1.532646,0.006249,-1.221010
124997,0.246357,-1.067017,-0.680876,-0.237983,-1.404989,0.002839
124998,-1.071003,2.578589,0.708297,-1.433536,1.417487,1.226689


# Customer Segmentation (Behavior-based RFM)

본 데이터는 단순 구매 기반 RFM이 아니라 **사용 행동 기반 Engagement RFM**을 기반으로 고객을 세그먼트화한다.
사용량, 플랫폼 참여도, 구독 리스크, 고객 불만 신호를 종합하여 고객 유형을 다음과 같이 정의할 수 있다.

---

## 1. Power Users (핵심 사용자)

**특징**

- activity_score ↑
- adjusted_frequency ↑
- Monetary ↑
- Engagement ↑
- subscription_risk ↓
- support_pressure ↓

**설명**

플랫폼을 가장 활발하게 사용하는 핵심 고객층이다.
음악 소비량이 많고, 플레이리스트 생성 및 친구 활동 등 플랫폼 참여도가 높으며 서비스 만족도가 높은 사용자이다.

---

## 2. Passive Heavy Listeners (소비 중심 사용자)

**특징**

- activity_score 중간
- adjusted_frequency ↑
- Monetary ↑
- Engagement ↓
- subscription_risk ↓

**설명**

음악을 많이 듣지만 플랫폼 기능 사용은 적은 사용자이다.
플레이리스트 생성, 친구 공유 등 소셜 활동은 거의 하지 않고 콘텐츠 소비에 집중하는 유형이다.

---

## 3. Social Explorers (소셜 활동 사용자)

**특징**

- activity_score ↑
- adjusted_frequency 중간
- Monetary 중간
- Engagement ↑
- subscription_risk 낮음

**설명**

플랫폼 내 커뮤니티 활동이 활발한 사용자이다.
플레이리스트 생성, 친구 공유 등 플랫폼 기능을 적극적으로 활용하는 특징이 있다.

---

## 4. Casual Users (가벼운 사용자)

**특징**

- activity_score ↓
- adjusted_frequency ↓
- Monetary ↓
- Engagement ↓
- subscription_risk 낮음

**설명**

플랫폼 사용 빈도가 낮은 사용자이다.
음악 소비량과 플랫폼 활동 모두 낮으며 서비스에 대한 의존도가 낮은 그룹이다.

---

## 5. At-Risk Subscribers (이탈 위험 사용자)

**특징**

- activity_score ↓
- adjusted_frequency ↓
- Monetary ↓
- subscription_risk ↑
- support_pressure 중간

**설명**

구독을 중단할 가능성이 높은 사용자이다.
서비스 이용량이 감소하고 구독 일시중단 횟수가 증가하는 패턴을 보인다.

---

## 6. Frustrated Users (불만 사용자)

**특징**

- support_pressure ↑
- subscription_risk ↑
- activity_score 중간

**설명**

서비스 이용 과정에서 문제나 불만을 경험한 사용자이다.
고객센터 문의가 많고 서비스 만족도가 낮아 이탈 가능성이 높은 그룹이다.

---

## 7. New / Exploring Users (탐색 단계 사용자)

**특징**

- activity_score ↑
- adjusted_frequency ↓
- Monetary ↓
- Engagement 중간
- subscription_risk 낮음

**설명**

최근 서비스를 사용하기 시작한 사용자이다.
플랫폼 기능을 탐색하는 단계이며 향후 핵심 사용자로 성장할 가능성이 있는 그룹이다.

---